In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/annotated/checkpoints/pre_cell_4.pickle

trying: ['lineitem']


me:  1
trying: ['tpch_parent']
me:  0
trying: ['load_supplier']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['supplier']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_nation']
me:  4
trying: ['load_orders']
me:  2
trying: ['orders']


me:  2
trying: ['nation']
me:  4
trying: ['pd']
me:  0


Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_nation
Declaring variable nation


In [4]:
%%RecordEvent
%%time
### cell 4 ###

# Filter lineitems where receipt date > commit date
mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
fi = lineitem[mask_date]

# Compute unique supplier counts per order
orig_count = (
    lineitem
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'orig_count'})
)
after_count = (
    fi
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'after_count'})
)

# Identify valid orders (orig_count >1 and after_count ==1)
valid_orders = orig_count.merge(after_count, on='L_ORDERKEY', how='inner')
valid_orders = valid_orders[(valid_orders.orig_count > 1) & (valid_orders.after_count == 1)]['L_ORDERKEY']

# Restrict to orders with status 'F'
f_orders = orders[orders.O_ORDERSTATUS == 'F']['O_ORDERKEY']
valid_orders = valid_orders[valid_orders.isin(f_orders)]

# Pre-filter Saudi Arabian suppliers
supplier_sa = supplier.merge(
    nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
    left_on='S_NATIONKEY',
    right_on='N_NATIONKEY',
    how='inner'
)[['S_SUPPKEY', 'S_NAME']]

# Final aggregation: count waiting lineitems per Saudi supplier
total = (
    lineitem[mask_date & lineitem.L_ORDERKEY.isin(valid_orders)]
    .merge(supplier_sa, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size': 'NUMWAIT'})
    .sort_values(['NUMWAIT', 'S_NAME'], ascending=[False, True])
)

CPU times: user 164 ms, sys: 79.9 ms, total: 244 ms
Wall time: 257 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q21/rewritten/o4_mini_high/checkpoints/post_cell_4_try_4.pickle

migration speed (bps): 798559462.9663569
---------------------------
variables to migrate:
load_lineitem 144
lineitem 48
orig_count 48
STORAGE_OPTS 64
tpch_parent 54
fi 48
pd 72
load_supplier 144
total 48
DATA_ROOT 80
supplier 48
after_count 48
supplier_sa 48
valid_orders 48
nation 48
load_nation 144
load_orders 144
orders 48
f_orders 48
mask_date 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q21/rewritten/o4_mini_high/checkpoints/post_cell_4_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['lineitem', 'supplier', 'orders']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['fi', 'f_orders', 'supplier_sa', 'orig_count', 'valid_orders', 'total', 'mask_date', 'after_count']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q21/opt_cell_exec_info_4_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[4], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/annotated/checkpoints/post_cell_4.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
